# Практична робота №1 — Математична модель складної системи та класифікація задач оптимізації

Достатньо змінити **лише номер варіанта** в клітинці нижче — весь інший код підхопить твою функцію, область аналізу та назву об'єкта автоматично.

In [1]:
import numpy as np
import sympy as sp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

## 1. Таблиця варіантів

Кожен рядок — це (a, b, c, d, e, k, x1_min, x1_max, x2_min, x2_max, назва об'єкта) для функції виду
`f(x1, x2) = a*x1^2 + b*x2^2 - c*x1*x2 - d*x1 - e*x2 + k`.

Просто вкажи свій номер у `VARIANT` — усе решта порахується саме для нього.

In [2]:
VARIANTS = {
    1:  (4, 3, 2, 3, 4, 6,   -1, 5, -1, 5, "логістичний склад"),
    2:  (2, 4, 3, 4, 5, 7,    0, 6, -2, 6, "ділянка складання"),
    3:  (5, 5, 4, 5, 6, 8,   -2, 7, -1, 7, "вузол зв'язку"),
    4:  (3, 6, 1, 6, 7, 9,   -1, 4, -2, 8, "ферма серверів"),
    5:  (1, 7, 2, 7, 8, 10,   0, 5, -1, 4, "ремонтна дільниця"),
    6:  (4, 2, 3, 8, 9, 11,  -2, 6, -2, 5, "транспортний парк"),
    7:  (2, 3, 4, 9, 10, 12, -1, 7, -1, 6, "енергомережа"),
    8:  (5, 4, 1, 10, 11, 13, 0, 4, -2, 7, "контакт-центр"),
    9:  (3, 5, 2, 11, 12, 5, -2, 5, -1, 8, "лінія розливу"),
    10: (1, 6, 3, 2, 13, 6,  -1, 6, -2, 4, "ділянка сортування"),
    11: (4, 7, 4, 3, 14, 7,   0, 7, -1, 5, "логістичний склад"),
    12: (2, 2, 1, 4, 3, 8,   -2, 4, -2, 6, "ділянка складання"),
    13: (5, 3, 2, 5, 4, 9,   -1, 5, -1, 7, "вузол зв'язку"),
    14: (3, 4, 3, 6, 5, 10,   0, 6, -2, 8, "ферма серверів"),
    15: (1, 5, 4, 7, 6, 11,  -2, 7, -1, 4, "ремонтна дільниця"),
    16: (4, 6, 1, 8, 7, 12,  -1, 4, -2, 5, "транспортний парк"),
    17: (2, 7, 2, 9, 8, 13,   0, 5, -1, 6, "енергомережа"),
    18: (5, 2, 3, 10, 9, 5,  -2, 6, -2, 7, "контакт-центр"),
    19: (3, 3, 4, 11, 10, 6, -1, 7, -1, 8, "лінія розливу"),
    20: (1, 4, 1, 2, 11, 7,   0, 4, -2, 4, "ділянка сортування"),
    21: (4, 5, 2, 3, 12, 8,  -2, 5, -1, 5, "логістичний склад"),
    22: (2, 6, 3, 4, 13, 9,  -1, 6, -2, 6, "ділянка складання"),
    23: (5, 7, 4, 5, 14, 10,  0, 7, -1, 7, "вузол зв'язку"),
    24: (3, 2, 1, 6, 3, 11,  -2, 4, -2, 8, "ферма серверів"),
    25: (1, 3, 2, 7, 4, 12,  -1, 5, -1, 4, "ремонтна дільниця"),
    26: (4, 4, 3, 8, 5, 13,   0, 6, -2, 5, "транспортний парк"),
    27: (2, 5, 4, 9, 6, 5,   -2, 7, -1, 6, "енергомережа"),
    28: (5, 6, 1, 10, 7, 6,  -1, 4, -2, 7, "контакт-центр"),
    29: (3, 7, 2, 11, 8, 7,   0, 5, -1, 8, "лінія розливу"),
    30: (1, 2, 3, 2, 9, 8,   -2, 6, -2, 4, "ділянка сортування"),
}

# <<< ЗМІНЮЄШ ЛИШЕ ЦЕ ЧИСЛО ПІД СВІЙ НОМЕР У ЖУРНАЛІ ГРУПИ >>>
VARIANT = 1

## 2. Побудова моделі за варіантом

Тут із таблиці дістаються коефіцієнти й область аналізу, і символьно (через SymPy) збирається сама функція `f`.

In [3]:
a, b, c, d, e, k, x1_lo, x1_hi, x2_lo, x2_hi, obj_name = VARIANTS[VARIANT]

x1, x2 = sp.symbols('x1 x2', real=True)
f = a*x1**2 + b*x2**2 - c*x1*x2 - d*x1 - e*x2 + k

print(f"Варіант {VARIANT} — об'єкт: {obj_name}")
print(f"f(x1, x2) = {f}")
print(f"Область аналізу: x1 ∈ [{x1_lo}, {x1_hi}], x2 ∈ [{x2_lo}, {x2_hi}]")

Варіант 1 — об'єкт: логістичний склад
f(x1, x2) = 4*x1**2 - 2*x1*x2 - 3*x1 + 3*x2**2 - 4*x2 + 6
Область аналізу: x1 ∈ [-1, 5], x2 ∈ [-1, 5]


## 3. Градієнт і матриця Гессе

`sp.diff(f, v)` рахує часткову похідну f по змінній v — це і є компоненти градієнта.
`sp.hessian(f, (x1, x2))` одразу будує матрицю других похідних (Гессе), не треба рахувати вручну кожен елемент.

In [4]:
grad = [sp.diff(f, v) for v in (x1, x2)]
H = sp.hessian(f, (x1, x2))
print("grad f =", grad)
print("H =", H.tolist())

grad f = [8*x1 - 2*x2 - 3, -2*x1 + 6*x2 - 4]
H = [[8, -2], [-2, 6]]


## 4. Стаціонарна точка

Розв'язуємо систему `grad f = 0` — це необхідна умова екстремуму (перший порядок). `sp.solve` повертає координати точки-кандидата, а потім підставляємо її назад у `f`, щоб дізнатись значення функції в цій точці (`f.subs(crit)`).

In [5]:
crit = sp.solve(grad, [x1, x2], dict=True)[0]
print("стаціонарна точка:", crit)
print("f* =", sp.simplify(f.subs(crit)))

стаціонарна точка: {x1: 13/22, x2: 19/22}
f* = 149/44


## 5. Перевірка опуклості (критерій Сильвестра) і число зумовленості

- `H[:k, :k].det()` для k=1,2 — це кутові мінори Δ₁ і Δ₂. Якщо обидва додатні — H додатно визначена, функція опукла, і знайдена стаціонарна точка є **єдиним глобальним мінімумом**.
- `H.eigenvals()` — власні числа матриці; той самий висновок про опуклість можна зробити, якщо всі вони додатні (λᵢ > 0) — це просто альтернативний спосіб перевірки, який виводимо для звірки.
- `cond(H) = λmax/λmin` — число зумовленості: чим воно ближче до 1, тим "кругліші" лінії рівня і тим швидше збігаються градієнтні методи; велике значення означає витягнутий "яр" і повільну збіжність.

In [6]:
minors = [H[:m, :m].det() for m in (1, 2)]
eigenvals = [sp.nsimplify(v) for v in H.eigenvals()]
eigen_floats = sorted(float(v) for v in eigenvals)
cond_H = eigen_floats[-1] / eigen_floats[0]

print("кутові мінори:", minors)
print("власні числа:", eigenvals)
print("опукла (усі мінори > 0):", all(m > 0 for m in minors))
print(f"число зумовленості cond(H) = {cond_H:.3f}")

кутові мінори: [8, 44]
власні числа: [7 - sqrt(5), sqrt(5) + 7]
опукла (усі мінори > 0): True
число зумовленості cond(H) = 1.939


## 6. Чи належить знайдена точка заданій області

За завданням мінімум шукається на всьому просторі ℝ² (безумовна оптимізація), але область аналізу x1/x2 задана з таблиці — перевіряємо, чи потрапляє знайдений x* в межі цієї області (якщо ні — на графіку мінімум буде видно за краєм, і варто окремо обговорити мінімум на межі області у висновках).

In [7]:
x1_star, x2_star = float(crit[x1]), float(crit[x2])
in_bounds = (x1_lo <= x1_star <= x1_hi) and (x2_lo <= x2_star <= x2_hi)
print(f"x* = ({x1_star:.4f}, {x2_star:.4f})")
print("належить заданій області:", in_bounds)

x* = (0.5909, 0.8636)
належить заданій області: True


## 7. Лінії рівня

`sp.lambdify` перетворює символьну функцію на звичайну numpy-функцію (щоб рахувати її швидко на сітці значень, а не символьно). `np.meshgrid` будує сітку x1/x2 у межах саме твоєї області з таблиці, `ax.contour` малює лінії рівня, зірочкою позначена знайдена точка мінімуму.

In [8]:
fn = sp.lambdify((x1, x2), f, 'numpy')
X, Y = np.meshgrid(np.linspace(x1_lo, x1_hi, 300), np.linspace(x2_lo, x2_hi, 300))
Z = fn(X, Y)

In [9]:
fig, ax = plt.subplots(figsize=(6, 5))
cs = ax.contour(X, Y, Z, levels=20)
ax.clabel(cs, inline=True, fontsize=7)
ax.plot(x1_star, x2_star, 'r*', ms=14, label='точка мінімуму')
ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$'); ax.legend(); ax.grid(alpha=.3)
ax.set_title(f'Варіант {VARIANT} ({obj_name}) — лінії рівня цільової функції')
fig.tight_layout(); fig.savefig('contour.png', dpi=150)
fig

<Figure size 600x500 with 1 Axes>